<a href="https://colab.research.google.com/github/Noob0005/Colab-notebooks/blob/main/notebooks/OmniVoice_Studio_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VoiceStudio on Google Colab

Run the full [VoiceStudio](https://github.com/debpalash/VoiceStudio) app — voice cloning, voice design, video dubbing, and TTS in 646 languages — on a free Colab GPU, web UI included.

**Before you start:** `Runtime → Change runtime type → T4 GPU` (the notebook also works on CPU, but generation is much slower).

Run the cells top to bottom. Every cell is idempotent — re-running after a hiccup is always safe.

**Contents**

- **Part 1 — Setup & launch (cells 1-7):** GPU check, install, backend launch, open the web UI, API smoke test.
- **Part 2 — Feature tour (cells 8-20):** each major feature as a self-contained API demo with inline playback — multilingual TTS, voice cloning, voice design, voice profiles, transcription, AI-watermark detection, the OpenAI-compatible API, a multi-voice story, a chaptered audiobook, video dubbing, and vocal isolation.
- **Part 3 — Troubleshooting.**

**How this notebook works (and why):**

| Piece | Approach | Why |
|---|---|---|
| Backend install | `uv pip install --system .` | Mirrors the official Docker image: installs into Colab's Python and keeps Colab's preinstalled CUDA PyTorch when it matches our pinned torch, instead of re-downloading multi-GB wheels into a venv. The pins (`deploy/torch-constraints.txt`) are passed explicitly so torch, torchaudio and torchvision cannot drift apart (#1357). |
| Web UI | Built in-notebook with `bun` (~2 min, once per session) | Official releases ship desktop installers only — there is no prebuilt standalone web bundle to download. The backend serves the built SPA itself from `frontend/dist`. |
| Opening the app | Colab's built-in kernel port proxy | No third-party tunnel binaries; the URL is private to your Google session. (A `cloudflared` public-URL alternative is documented in the launch cell.) |

Issues with this notebook are VoiceStudio issues — report them at [debpalash/VoiceStudio/issues](https://github.com/debpalash/VoiceStudio/issues).


## Part 1 — Setup & launch


In [2]:
# ── 1. GPU check ────────────────────────────────────────────────────────────
# Confirms the runtime has an NVIDIA GPU. Everything still works on CPU, but
# a single sentence can take minutes instead of seconds.
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
    print("GPU detected — you're good to go.")
else:
    print("=" * 74)
    print("WARNING: no NVIDIA GPU in this runtime.")
    print("Fix: menu bar -> Runtime -> Change runtime type -> T4 GPU,")
    print("then re-run this notebook from the top.")
    print("(Continuing anyway works, but generation will be very slow on CPU.)")
    print("=" * 74)


Mon Nov  2 17:33:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# ── 2. Clone + install (idempotent; first run ~5-8 min) ─────────────────────
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/debpalash/VoiceStudio"
REPO_DIR = "/content/VoiceStudio"
BUN = "/root/.bun/bin/bun"

# Generous network timeouts: Colab -> PyPI is usually fast, but model/wheel
# CDNs occasionally stall and uv's default timeout is short.
os.environ.setdefault("UV_HTTP_TIMEOUT", "300")

def run(cmd, *, cwd=None, what=""):
    """Stream a command's output; stop the cell with an actionable message on failure."""
    shown = cmd if isinstance(cmd, str) else " ".join(cmd)
    print(f"\n$ {shown}")
    rc = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str)).returncode
    if rc != 0:
        raise SystemExit(
            f"\nFAILED: {what or shown} (exit code {rc}).\n"
            "Scroll up for the underlying error. Re-running this cell is safe.\n"
            "Still stuck? Open an issue with the output above:\n"
            "  https://github.com/debpalash/VoiceStudio/issues"
        )

# 2a. Clone the repo (reused if already present)
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    # Bring a clone from an earlier session up to date: the cells below pass files
    # that older clones lack (deploy/uv-overrides.txt), and a stale clone would
    # otherwise install old code. A changed checkout also rebuilds the web UI (2d).
    print(f"Repo already present at {REPO_DIR} — updating it.")
    head = lambda: subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout
    before = head()
    run(["git", "-C", REPO_DIR, "pull", "--ff-only"],
        what=f"git pull in {REPO_DIR} (if you changed files there, delete the folder and re-run this cell)")
    if head() != before:
        shutil.rmtree(os.path.join(REPO_DIR, "frontend", "dist"), ignore_errors=True)
else:
    run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], what="git clone")

# 2b. System packages (ffmpeg: audio/video processing; libsndfile1: soundfile)
run("apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1", what="apt-get install")

# 2c. bun — builds the web UI (see the intro cell for why we build it here)
if not os.path.exists(BUN):
    run("curl -fsSL https://bun.sh/install | bash", what="bun installer")
os.environ["PATH"] = os.path.dirname(BUN) + os.pathsep + os.environ["PATH"]
run([BUN, "--version"], what="bun version check")

# 2d. Build the frontend (skipped once frontend/dist/index.html exists)
dist_index = os.path.join(REPO_DIR, "frontend", "dist", "index.html")
if os.path.exists(dist_index):
    print("frontend/dist already built — skipping.")
else:
    # The repo is a bun workspace: install at the repo root, build the Electron web renderer.
    run([BUN, "install", "--frozen-lockfile"], cwd=REPO_DIR, what="bun install (workspace)")
    run([BUN, "run", "--cwd", "electron", "build:web"], cwd=REPO_DIR, what="Electron web renderer build (vite)")
    if not os.path.exists(dist_index):
        raise SystemExit(
            "The frontend build finished but frontend/dist/index.html is missing —\n"
            "scroll up for vite errors, then re-run this cell."
        )

# 2e. Backend dependencies — same command the official Docker image uses.
if not shutil.which("uv"):
    run([sys.executable, "-m", "pip", "install", "-q", "uv"], what="pip install uv")
# --constraint: `uv pip install` ignores pyproject's `[tool.uv]
# constraint-dependencies` (a project-API setting), so without it torch,
# torchaudio and torchvision resolve on their bare lower bounds and Colab's
# preinstalled torchvision can be left behind a newer torch -- which fails at
# import with "operator torchvision::nms does not exist" (#1357).
# --override: `[tool.uv] override-dependencies` (lifting gradio's pillow<13 cap
# for Pillow's security releases) only applies from the project directory with
# a recent uv; pass it explicitly (#3275).
run(["uv", "pip", "install", "--system", "--no-cache",
     "--constraint", "deploy/torch-constraints.txt",
     "--override", "deploy/uv-overrides.txt", "."],
    cwd=REPO_DIR, what="backend install (uv pip install --system .)")

# 2f. cuDNN 8 side-install for CTranslate2 (WhisperX ASR on GPU). PyTorch 2.8+
# ships cuDNN 9; scripts/setup.py places cuDNN 8 libs where the backend
# preloads them from (<repo>/.venv/.../cudnn8_compat). We create the .venv
# directory only so the script has its expected target — no actual venv is
# used on Colab. Non-fatal: without it, transcription falls back to the
# torch-native Whisper backend automatically.
os.makedirs(os.path.join(REPO_DIR, ".venv"), exist_ok=True)
run([sys.executable, os.path.join(REPO_DIR, "scripts", "setup.py")], what="cuDNN 8 compat setup")

# 2g. Sanity check in a fresh interpreter (this kernel may hold a stale torch).
# Imports the backend's own model stack, not just torch — Colab's system
# Python mixes preinstalled and freshly-resolved wheels, and a torchaudio or
# transformers that can't load together only shows up when the model module is
# imported (#1229). Catching it here beats a 5-minute health timeout in cell 5.
run([sys.executable, "-c",
     # Versions FIRST: if the model-stack import below fails, the cell output
     # still shows what was actually installed, which is the single most
     # useful line for diagnosing a Colab environment (#1229).
     "import torch, torchaudio, uvicorn, fastapi, transformers; "
     "print(f'torch {torch.__version__}, torchaudio {torchaudio.__version__}, "
     "transformers {transformers.__version__}, "
     "CUDA available: {torch.cuda.is_available()}'); "
     "from omnivoice.models.omnivoice import OmniVoice; "
     "from transformers import HiggsAudioV2TokenizerModel; "
     "print('Install OK - backend model stack imports cleanly')"],
    cwd=REPO_DIR, what="import sanity check")



$ git clone --depth 1 https://github.com/debpalash/VoiceStudio /content/VoiceStudio

$ apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1

$ curl -fsSL https://bun.sh/install | bash

$ /root/.bun/bin/bun --version

$ /root/.bun/bin/bun install --frozen-lockfile

$ /root/.bun/bin/bun run --cwd electron build:web

$ uv pip install --system --no-cache --constraint deploy/torch-constraints.txt --override deploy/uv-overrides.txt .

$ /usr/bin/python3 /content/VoiceStudio/scripts/setup.py

$ /usr/bin/python3 -c import torch, torchaudio, uvicorn, fastapi, transformers; print(f'torch {torch.__version__}, torchaudio {torchaudio.__version__}, transformers {transformers.__version__}, CUDA available: {torch.cuda.is_available()}'); from omnivoice.models.omnivoice import OmniVoice; from transformers import HiggsAudioV2TokenizerModel; print('Install OK - backend model stack imports cleanly')


In [4]:
# ── 3. (Optional) Hugging Face token ────────────────────────────────────────
# Only needed for GATED models — e.g. pyannote speaker diarization, used by
# video dubbing for multi-speaker detection. TTS, voice cloning, and voice
# design all work WITHOUT a token, so feel free to skip this cell.
#
# To use one: accept the model terms at
#   https://huggingface.co/pyannote/speaker-diarization-3.1
# then add a Colab Secret named HF_TOKEN (key icon in the left sidebar),
# toggle "Notebook access" ON, and re-run this cell.
import os

try:
    from google.colab import userdata
    _token = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = _token
    print("HF_TOKEN loaded from Colab Secrets — gated models (diarization) enabled.")
except Exception:
    print("No HF_TOKEN Colab Secret found — skipping. (Everything except gated "
          "diarization models works without it.)")


HF_TOKEN loaded from Colab Secrets — gated models (diarization) enabled.


In [4]:
# ── 4. (Optional, recommended) Pre-download the default models ──────────────
# The default TTS engine fetches k2-fsa/OmniVoice (a few GB) on its very first
# generation. Downloading it up front makes the first request fast instead of
# a several-minute stall. Safe to re-run: already-downloaded files are reused.
from huggingface_hub import snapshot_download

path = snapshot_download("k2-fsa/OmniVoice")
print("Default TTS model cached at:", path)

# The default speech-to-text model (~3 GB). The app only transcribes a new
# voice's reference clip with a model that is already installed, so fetching
# it now gives clones saved in the web UI a transcript from the start.
# Cell 12a then only checks the cache.
!hf download Systran/faster-whisper-large-v3


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Default TTS model cached at: /root/.cache/huggingface/hub/models--k2-fsa--OmniVoice/snapshots/c5fdb5ccb189668d56333f77ba2629f4cd7535f4
Hint: A new version of huggingface_hub (2.4.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0% 0/7 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/3.09G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/3.09G [00:00<?, ?B/s]

Fetching 7 files:  14% 1/7 [00:00<00:02,  2.81it/s]
Reconstructing (incomplete total...):   0% 1.52k/3.09G [00:00<200:01:26, 4.29kB/s]
Reconstructing (incomplete total...):   0% 3.91k/3.09G [00:00<200:05:34, 4.29kB/s]
Reconstructing (incomplete total...):   0% 3.91k/3.09G [00:00<200:05:34, 4.29kB/s]
Reconstructing (incomplete total...):   0% 4.25k/3.09G [00

In [29]:
import os, re, secrets, subprocess, sys, time, json, urllib.request
from pathlib import Path

REPO_DIR = "/content/VoiceStudio"
PORT = 3900
LOG = "/content/omnivoice_backend.log"
CF_BIN = "/usr/local/bin/cloudflared"
CF_LOG = "/content/cloudflared.log"
KEY_FILE = Path("/content/omnivoice_api_key.txt")

# One key for the whole session, shared by the browser and the test cells
if not KEY_FILE.exists():
    KEY_FILE.write_text(secrets.token_urlsafe(24))
API_KEY = KEY_FILE.read_text().strip()

# Stop the old backend and tunnel
subprocess.run("kill -9 $(lsof -t -i:3900) 2>/dev/null; pkill -x cloudflared 2>/dev/null; sleep 2", shell=True)

# Install cloudflared once
if not os.path.exists(CF_BIN):
    subprocess.run(
        f"wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {CF_BIN} && chmod +x {CF_BIN}",
        shell=True, check=True)

# Start the tunnel and read its public hostname
open(CF_LOG, "w").close()
subprocess.Popen([CF_BIN, "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
                 stdout=open(CF_LOG, "a"), stderr=subprocess.STDOUT)
host = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://((?!api\.)[a-z0-9-]+\.trycloudflare\.com)", open(CF_LOG, errors="replace").read())
    if m:
        host = m.group(1)
        break
if not host:
    raise SystemExit(f"No tunnel URL after 60s. Check {CF_LOG}")
public = f"https://{host}"

# Start the backend, allowing only this tunnel's host
env = os.environ.copy()
env.update({
    "OMNIVOICE_SERVER_MODE": "1",
    "OMNIVOICE_DATA_DIR": "/content/omnivoice_data",
    "OMNIVOICE_API_KEY": API_KEY,
    "OMNIVOICE_ALLOWED_HOSTS": f"{host},localhost,127.0.0.1",
    "OMNIVOICE_ALLOWED_ORIGINS": public,
    "PYTHONUNBUFFERED": "1",
})
subprocess.Popen([sys.executable, "-m", "uvicorn", "main:app", "--app-dir", "backend",
                  "--host", "127.0.0.1", "--port", str(PORT)],
                 cwd=REPO_DIR, env=env, stdout=open(LOG, "ab"), stderr=subprocess.STDOUT)

# Wait for /health
health = None
deadline = time.time() + 300
while time.time() < deadline:
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5) as r:
            health = json.load(r)
            break
    except Exception:
        time.sleep(3)

print("Backend:", health or "NOT HEALTHY - run: !tail -n 40 /content/omnivoice_backend.log")
print("\nOpen in a NEW Incognito window:", public)
print("API key:", API_KEY)

Backend: {'status': 'ok', 'device': 'cuda (Tesla T4)', 'version': '0.5.7'}

Open in a NEW Incognito window: https://millennium-precisely-retailer-sacrifice.trycloudflare.com
API key: 5su1VguMNwqV9hpgC24V5KdGNfZEDhaU


In [22]:
# ── 7. Smoke test: generate speech through the API ──────────────────────────
# Proves the whole stack end-to-end without touching the UI: /health, then a
# real TTS generation played inline. POST /generate returns the WAV directly.
import json
import urllib.error
import urllib.parse
import urllib.request

BASE = "http://127.0.0.1:3900"

with urllib.request.urlopen(f"{BASE}/health", timeout=10) as r:
    print("GET /health ->", json.load(r))

data = urllib.parse.urlencode({
    "text": "VoiceStudio is up and running on Google Colab.",
}).encode()
print("Generating... (first-ever generation loads the model — if you skipped "
      "cell 4 it also downloads it, which can take several minutes)")
try:
    with urllib.request.urlopen(
        urllib.request.Request(f"{BASE}/generate", data=data), timeout=1800
    ) as r:
        wav = r.read()
        print(f"OK — {len(wav)} bytes, duration {r.headers.get('X-Audio-Duration')}s, "
              f"generated in {r.headers.get('X-Gen-Time')}s (seed {r.headers.get('X-Seed')})")
except urllib.error.HTTPError as e:
    detail = e.read().decode("utf-8", errors="replace")
    try:
        payload = json.loads(detail)
    except (ValueError, TypeError):
        payload = None
    err_detail = payload.get("detail") if isinstance(payload, dict) else None
    if isinstance(err_detail, dict) and err_detail.get("code") == "model_licence_required":
        models = err_detail.get("models") or []
        repos = ", ".join(m.get("repo_id", "") for m in models if isinstance(m, dict) and m.get("repo_id"))
        msg = err_detail.get("message") or "Model licence acceptance required."
        raise SystemExit(
            "Generation failed: HTTP 403 - model licence not accepted.\n"
            + (f"Required models: {repos}\n" if repos else "")
            + f"{msg}\n"
            "Open the app from cell 6 (this Colab session's own backend keeps its own "
            "licence acceptance), review and accept the listed licences in Model Catalogue, "
            "then retry. Accepting them in a desktop install does not apply here. This is not a download or GPU failure."
        )
    raise SystemExit(
        f"Generation failed: HTTP {e.code}\n{detail}\n"
        "Check /content/omnivoice_backend.log for the full trace; on a fresh\n"
        "session the usual cause is an interrupted model download — re-run\n"
        "cell 4, then this cell."
    )

OUT = "/content/omnivoice_smoke.wav"
with open(OUT, "wb") as f:
    f.write(wav)

from IPython.display import Audio, display
display(Audio(OUT))


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



GET /health -> {'status': 'ok', 'device': 'cuda (Tesla T4)', 'version': '0.5.7'}
Generating... (first-ever generation loads the model — if you skipped cell 4 it also downloads it, which can take several minutes)
Traceback (most recent call last):
  File "/tmp/ipykernel_1225/359548941.py", line 20, in <cell line: 0>
    with urllib.request.urlopen(
         ~~~~~~~~~~~~~~~~~~~~~~^
        urllib.request.Request(f"{BASE}/generate", data=data), timeout=1800
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ) as r:
    ^
  File "/usr/lib/python3.13/urllib/request.py", line 189, in urlopen
    return opener.open(url, data, timeout)
           ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/urllib/request.py", line 495, in open
    response = meth(req, response)
  File "/usr/lib/python3.13/urllib/request.py", line 604, in http_response
    response = self.parent.error(
        'http', request, response, code, msg, hdrs)
  File "/usr/lib/python3.13/ur

TypeError: object of type 'NoneType' has no len()

## Part 2 — Feature tour

Each cell below demos one major feature straight through the backend's HTTP API — the same API the desktop app and the web UI use. Every cell is self-contained (it regenerates anything it needs), plays its result inline, and states its expected runtime honestly. Requirements: the backend from cell 5 must be running, and **run cell 8 (helpers) once first**.

Endpoints used, per section: `/generate` (9-12), `/design/describe` (11), `/profiles` (12), `/transcribe` (13), `/watermark/status` + `/watermark/detect` (14), `/v1/audio/*` (15), `/longform/render` (16), `/audiobook` (17), `/dub/*` + `/jobs/*` (18-19).


### 8. Feature-tour helpers

Shared plumbing for the tour: a `/generate` wrapper that saves WAVs under `/content/omnivoice_demos/`, inline playback, and loud failures that point at the backend log. Run this once before any feature cell.


In [ ]:
# ── 8. Feature-tour helpers (run once before cells 9-20) ────────────────────
import json
import os

import requests
from IPython.display import Audio, display

BASE = "http://127.0.0.1:3900"
DEMO_DIR = "/content/omnivoice_demos"
LOG_HINT = ("Backend log: /content/omnivoice_backend.log — attach it when "
            "reporting: https://github.com/debpalash/VoiceStudio/issues")
os.makedirs(DEMO_DIR, exist_ok=True)

# Shared sample lines (cells 9/10/12/13 reuse these to stay self-contained).
EN_TEXT = "This sentence was generated locally, on a free cloud graphics card."
ES_TEXT = "La clonación de voz ya no necesita la nube: todo sucede aquí mismo."
BN_TEXT = "আমার কণ্ঠস্বর এখন ছয়শো ছেচল্লিশটি ভাষায় কথা বলতে পারে।"

def check_backend():
    try:
        requests.get(f"{BASE}/health", timeout=5).raise_for_status()
    except Exception:
        raise SystemExit("The backend is not answering — run cell 5 first. " + LOG_HINT)

def _error_detail(resp):
    try:
        payload = resp.json()
    except (ValueError, TypeError, AttributeError):
        try:
            payload = json.loads(resp.text)
        except (ValueError, TypeError):
            return None
    if isinstance(payload, dict):
        detail = payload.get("detail")
        if isinstance(detail, dict):
            return detail
        if isinstance(payload.get("error"), dict):
            return payload["error"]
    return None

def api_fail(resp, doing):
    detail = _error_detail(resp)
    if isinstance(detail, dict) and detail.get("code") == "model_licence_required":
        models = detail.get("models") or []
        repos = ", ".join(m.get("repo_id", "") for m in models
                          if isinstance(m, dict) and m.get("repo_id"))
        msg = detail.get("message") or "Model licence acceptance required."
        raise SystemExit(
            f"{doing} failed: HTTP {resp.status_code} - model licence not accepted.\n"
            + (f"Required models: {repos}\n" if repos else "")
            + f"{msg}\n"
            "Open the app from cell 6 (this Colab session's own backend keeps its own "
            "licence acceptance), review and accept the listed licences in Model Catalogue, "
            "then retry. Accepting them in a desktop install does not apply here. This is not a download or GPU failure.\n"
            + LOG_HINT
        )
    raise SystemExit(
        f"{doing} failed: HTTP {resp.status_code}\n"
        f"{resp.text[:2000]}\n{LOG_HINT}"
    )

def speak(text, out_name, files=None, timeout=1800, **form):
    """POST /generate; save the returned WAV under DEMO_DIR; return (path, headers)."""
    check_backend()
    data = {"text": text, **{k: str(v) for k, v in form.items()}}
    r = requests.post(f"{BASE}/generate", data=data, files=files, timeout=timeout)
    if r.status_code != 200:
        api_fail(r, f"/generate ({out_name})")
    path = os.path.join(DEMO_DIR, out_name)
    with open(path, "wb") as f:
        f.write(r.content)
    return path, r.headers

def ensure_wav(out_name, text, **form):
    """Idempotent speak(): reuse the file if an earlier cell already made it."""
    path = os.path.join(DEMO_DIR, out_name)
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    return speak(text, out_name, **form)[0]

def play(path, label=None):
    if label:
        print(label)
    display(Audio(path))

print("Helpers ready. Demo artifacts will land in", DEMO_DIR)


### 9. Text-to-speech in many languages

Three short generations — English, Spanish, Bengali — through `POST /generate`, with the per-request seed and generation time from the response headers. No `language` parameter is passed: the engine autodetects from the text. Expected runtime: ~5-20 s per line on a warm T4 (the very first generation loads the model — minutes if cell 4 was skipped).


In [ ]:
# ── 9. TTS in many languages ────────────────────────────────────────────────
try:
    speak
except NameError:
    raise SystemExit("Run cell 8 (feature-tour helpers) first.")

for code, label, text in [("en", "English", EN_TEXT),
                          ("es", "Spanish", ES_TEXT),
                          ("bn", "Bengali", BN_TEXT)]:
    path, h = speak(text, f"tts_{code}.wav")
    print(f"{label}: seed={h.get('X-Seed')}  gen_time={h.get('X-Gen-Time')}s  "
          f"duration={h.get('X-Audio-Duration')}s")
    play(path)


### 10. Voice cloning (zero-shot)

Clone a voice from a single short reference clip — no training. To stay self-contained, the reference is the English WAV from cell 9 (regenerated if missing): the cloned line should come back in that same voice. A commented variant shows how to clone **your own** voice from an uploaded 3-10 s clip. Expected runtime: ~10-30 s on a warm T4.


In [ ]:
# ── 10. Voice cloning ───────────────────────────────────────────────────────
try:
    ensure_wav
except NameError:
    raise SystemExit("Run cell 8 (feature-tour helpers) first.")

ref_path = ensure_wav("tts_en.wav", EN_TEXT)
ref_text = EN_TEXT  # transcript of the reference clip (skips auto-transcription)

with open(ref_path, "rb") as ref:
    clone_path, h = speak(
        "And this is the very same voice, cloned from three seconds of audio.",
        "clone_demo.wav",
        files={"ref_audio": ("reference.wav", ref, "audio/wav")},
        ref_text=ref_text,
    )
print(f"Cloned in {h.get('X-Gen-Time')}s (seed {h.get('X-Seed')})")
play(ref_path, "Reference voice:")
play(clone_path, "Cloned voice, new sentence:")

# ── Clone YOUR OWN voice instead ──
# from google.colab import files
# uploaded = files.upload()          # pick a clean 3-10 s clip of one speaker
# my_clip = next(iter(uploaded))
# with open(my_clip, "rb") as ref:
#     my_clone, _ = speak("Any text you like, in my voice.", "my_clone.wav",
#                         files={"ref_audio": (my_clip, ref)})
#     # (no ref_text -> the backend transcribes the clip automatically)
# play(my_clone)


### 11. Voice design

Describe a voice in plain English; `POST /design/describe` maps the description onto the engine's design-parameter space (gender, age, pitch, accent, style) and returns a validated `instruct` string, which `/generate` then uses to synthesize that voice from nothing — no reference audio. The same sentence is rendered with two contrasting designs. Expected runtime: ~10-30 s per voice on a warm T4.


In [ ]:
# ── 11. Voice design ────────────────────────────────────────────────────────
try:
    speak
except NameError:
    raise SystemExit("Run cell 8 (feature-tour helpers) first.")

SENTENCE = "Every voice you hear in this notebook was invented by a description."
designs = [
    ("design_a.wav", "a deep, elderly male narrator with a low pitch and a british accent"),
    ("design_b.wav", "a young, energetic female voice with a high pitch"),
]

for out_name, description in designs:
    r = requests.post(f"{BASE}/design/describe", json={"description": description}, timeout=30)
    if r.status_code != 200:
        api_fail(r, "/design/describe")
    d = r.json()
    print(f'"{description}"')
    print(f"  -> instruct: {d['instruct']!r}")
    if d.get("unmatched"):
        print(f"  -> not mappable (ignored): {d['unmatched']}")
    path, h = speak(SENTENCE, out_name, instruct=d["instruct"])
    play(path, f"  gen_time={h.get('X-Gen-Time')}s")


### 12. Voice profiles — save, list, reuse

Voices become reusable assets: save the cell-9 narrator and the cell-11 designed voice as named profiles (`POST /profiles`), list them (`GET /profiles`), then generate by `profile_id` alone — no reference upload, no instruct string. Cells 16-17 reuse these two profiles. Idempotent: existing profiles with the same names are reused. Expected runtime: seconds, plus one ~10-30 s generation.


In [ ]:
# ── 12. Voice profiles ──────────────────────────────────────────────────────
try:
    ensure_wav
except NameError:
    raise SystemExit("Run cell 8 (feature-tour helpers) first.")

check_backend()

def ensure_profile(name, wav_path, ref_text):
    """Create a clone profile from a WAV, or reuse it if this name exists."""
    r = requests.get(f"{BASE}/profiles", timeout=30)
    if r.status_code != 200:
        api_fail(r, "GET /profiles")
    for p in r.json():
        if p.get("name") == name:
            print(f"Profile {name!r} already exists (id={p['id']}) — reusing.")
            return p["id"]
    with open(wav_path, "rb") as ref:
        r = requests.post(
            f"{BASE}/profiles",
            data={"name": name, "ref_text": ref_text, "kind": "clone"},
            files={"ref_audio": (os.path.basename(wav_path), ref, "audio/wav")},
            timeout=120,
        )
    if r.status_code != 200:
        api_fail(r, "POST /profiles")
    created = r.json()
    print(f"Created profile {name!r} (id={created['id']})")
    return created["id"]

NARRATOR_ID = ensure_profile(
    "Colab Narrator", ensure_wav("tts_en.wav", EN_TEXT), EN_TEXT)
GUEST_SENTENCE = "Every voice you hear in this notebook was invented by a description."
GUEST_ID = ensure_profile(
    "Colab Guest",
    ensure_wav("design_b.wav", GUEST_SENTENCE,
               instruct="female, young, high pitch"),
    GUEST_SENTENCE)

profiles = requests.get(f"{BASE}/profiles", timeout=30).json()
print(f"\n{len(profiles)} saved profile(s):")
for p in profiles:
    print(f"  {p['id']}  {p.get('kind', '?'):6s}  {p.get('name')}")

path, h = speak("Generating by profile id — no reference clip attached this time.",
                "profile_demo.wav", profile_id=NARRATOR_ID)
play(path, f"\nVoice of profile {NARRATOR_ID} (gen_time={h.get('X-Gen-Time')}s):")


### 12a. Install the transcription model

Cells 13 and 18 need a speech-to-text model on disk. The next cell downloads `Systran/faster-whisper-large-v3` from Hugging Face (roughly 3 GB), the model behind the default transcription backend. If you only want text-to-speech, you can skip it and stop after cell 12.

Running it again is cheap: the snapshot is re-checked, cached files are reused and only missing pieces are fetched. If cell 4 already pulled the model, this just verifies the cache. The notebook and the backend started in cell 5 read the same Hugging Face cache, so if you change cache settings afterwards, restart the backend. To use another ASR engine or model from the app, install it through Model Catalogue instead.

In [ ]:
# ── 12a. Install the transcription model ─────────────────────────────────────
# Flag that cells 13 and 18 check. Reset first, so a failed re-run cannot
# leave a stale True behind.
ASR_MODEL_READY = False

import huggingface_hub

# No local_dir on purpose: the shared Hugging Face cache is what the backend reads.
print("Checking / downloading Systran/faster-whisper-large-v3 (~3 GB on first run)...")
_asr_snapshot = huggingface_hub.snapshot_download(repo_id="Systran/faster-whisper-large-v3")
print("Transcription model is in the cache at:", _asr_snapshot)
ASR_MODEL_READY = True

# The backend prefers the CTranslate2 build of Whisper. It needs the cuDNN 8
# libraries from the earlier setup step, so verify the import now, while the
# download output above is still visible.
try:
    import ctranslate2  # noqa: F401
except Exception as exc:
    print("\nWARNING: ctranslate2 could not be imported "
          f"({type(exc).__name__}: {exc}).")
    print("The backend will fall back to its PyTorch Whisper default and download "
          "that other model separately (several more GB).")
    print("Re-run the cuDNN 8 setup cell above, then run this cell again.")

### 13. Transcription (speech-to-text)

This closes the loop on the cell-9 narration: the WAV goes through `POST /transcribe`, and the text that comes back should match what was spoken.

Run cell 12a first. The backend deliberately rejects transcription when its ASR model is missing and never downloads one on its own. The first transcription loads the installed model into memory, so expect a short pause.

In [ ]:
# ── 13. Transcription: TTS -> ASR round trip ────────────────────────────────
# Needs the model from cell 12a; stop early with a clear pointer if it is missing.
if not globals().get("ASR_MODEL_READY"):
    raise SystemExit("Run cell 12a (install the transcription model) first.")

try:
    ensure_wav
except NameError:
    raise SystemExit("Run cell 8 (feature-tour helpers) first.")

wav = ensure_wav("tts_en.wav", EN_TEXT)
print("Transcribing - the first run loads the installed ASR model, so it can take a moment...")
with open(wav, "rb") as f:
    r = requests.post(f"{BASE}/transcribe",
                      files={"audio": ("tts_en.wav", f, "audio/wav")},
                      timeout=1800)
if r.status_code != 200:
    api_fail(r, "POST /transcribe")
res = r.json()
print(f"Engine: {res.get('engine')}  "
      f"(audio {res.get('duration_s')}s, transcribed in {res.get('transcription_time_s')}s)")
print(f"  TTS input : {EN_TEXT}")
print(f"  ASR output: {res.get('text', '').strip()}")


### 14. AI-watermark detection

Every WAV VoiceStudio generates carries an inaudible AudioSeal watermark by default, so AI-generated audio can be identified later. `POST /watermark/detect` on a generated clip should report high confidence; the same check on a plain ffmpeg-made tone should not. Expected runtime: seconds (first call loads the small detector model).


In [ ]:
# ── 14. AI watermark: detect generated vs. plain audio ──────────────────────
import subprocess

try:
    ensure_wav
except NameError:
    raise SystemExit("Run cell 8 (feature-tour helpers) first.")

status = requests.get(f"{BASE}/watermark/status", timeout=30)
if status.status_code != 200:
    api_fail(status, "GET /watermark/status")
print("Watermark status:", status.json())

def detect(path):
    with open(path, "rb") as f:
        r = requests.post(f"{BASE}/watermark/detect",
                          files={"file": (os.path.basename(path), f, "audio/wav")},
                          timeout=300)
    if r.status_code != 200:
        api_fail(r, "POST /watermark/detect")
    return r.json()

generated = ensure_wav("tts_en.wav", EN_TEXT)
res = detect(generated)
print(f"\nVoiceStudio-generated clip: watermarked={res.get('is_watermarked')} "
      f"confidence={res.get('confidence')}")

plain = os.path.join(DEMO_DIR, "plain_tone.wav")
if not os.path.exists(plain):
    subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
                    "-f", "lavfi", "-i", "sine=frequency=440:duration=3", plain],
                   check=True)
res = detect(plain)
print(f"Plain (non-AI) tone:      watermarked={res.get('is_watermarked')} "
      f"confidence={res.get('confidence')}")


### 15. OpenAI-compatible API

Anything that speaks OpenAI's audio API works against `http://127.0.0.1:3900/v1` unchanged — here the official `openai` Python client does TTS (`/v1/audio/speech`) and STT (`/v1/audio/transcriptions`) against the local engine. No key is checked; `voice` also accepts your profile ids from cell 12. Expected runtime: seconds (warm models).


In [ ]:
# ── 15. OpenAI-compatible API with the official openai client ───────────────
import subprocess
import sys

try:
    ensure_wav
except NameError:
    raise SystemExit("Run cell 8 (feature-tour helpers) first.")

check_backend()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai"], check=True)
from openai import OpenAI

client = OpenAI(base_url=f"{BASE}/v1", api_key="none")  # any string — nothing checks it

speech = client.audio.speech.create(
    model="tts-1", voice="alloy", response_format="wav",
    input="This request thinks it is talking to OpenAI. It is not.",
)
oa_path = os.path.join(DEMO_DIR, "openai_compat.wav")
with open(oa_path, "wb") as f:
    f.write(speech.read())
play(oa_path, "POST /v1/audio/speech ->")

with open(oa_path, "rb") as f:
    tr = client.audio.transcriptions.create(model="whisper-1", file=f)
print("POST /v1/audio/transcriptions ->", tr.text.strip())


### 16. Multi-voice story

Two saved profiles perform a four-line scene through `POST /longform/render` — the Stories Editor's backend: per-span voices, inter-line pauses, and a single mixed MP3 out, with SSE progress along the way. Requires the profiles from cell 12. Expected runtime: ~1-2 min on a warm T4 (four TTS spans + encode).


In [ ]:
# ── 16. Multi-voice story via /longform/render ──────────────────────────────
import json as _json

try:
    NARRATOR_ID, GUEST_ID
except NameError:
    raise SystemExit("Run cell 12 (voice profiles) first — this cell reuses its two profiles.")

payload = {
    "default_voice": NARRATOR_ID,
    "format": "mp3",
    "chapters": [{
        "title": "A very short scene",
        "spans": [
            {"voice_id": NARRATOR_ID, "text": "Have you noticed we are running inside a notebook?", "pause_ms_after": 300},
            {"voice_id": GUEST_ID, "text": "I have. And nobody uploaded a single voice actor.", "pause_ms_after": 300},
            {"voice_id": NARRATOR_ID, "text": "Two profiles, four lines, one rendered story.", "pause_ms_after": 300},
            {"voice_id": GUEST_ID, "text": "Roll the credits.", "pause_ms_after": 0},
        ],
    }],
}

check_backend()
out_name = None
with requests.post(f"{BASE}/longform/render", json=payload,
                   stream=True, timeout=(10, 1800)) as r:
    if r.status_code != 200:
        api_fail(r, "POST /longform/render")
    for line in r.iter_lines(decode_unicode=True):
        if not line or not line.startswith("data:"):
            continue
        evt = _json.loads(line[len("data:"):].strip())
        etype = evt.get("type")
        if etype == "error":
            raise SystemExit(f"Story render failed: {evt}\n{LOG_HINT}")
        if etype == "done":
            out_name = evt["output"]
            print(f"Done: {evt.get('chapters')} chapter(s), {evt.get('duration_s')}s of audio")
        else:
            print(f"  [{etype}] {evt.get('title') or evt.get('text', '')!s:.60}")

if not out_name:
    raise SystemExit(f"Stream ended without a 'done' event. {LOG_HINT}")
story_path = os.path.join("/content/omnivoice_data/outputs", out_name)
play(story_path, f"Rendered story ({story_path}):")


### 17. Audiobook (chaptered m4b)

`POST /audiobook` takes a plain-text script — `# Heading` starts a chapter, `[pause 400ms]` inserts silence — and renders a chapterized, tagged `.m4b` with SSE progress (the same renderer handles resume after interruptions). Narrated by the cell-12 profile. Expected runtime: ~1-2 min on a warm T4 for this two-chapter miniature.


In [ ]:
# ── 17. Audiobook: script -> chaptered m4b ──────────────────────────────────
import base64
import json as _json

from IPython.display import HTML

try:
    NARRATOR_ID
except NameError:
    raise SystemExit("Run cell 12 (voice profiles) first — this cell narrates with its profile.")

SCRIPT = """# Chapter One: The Setup
It began, as most experiments do, with a borrowed graphics card. [pause 400ms] Nothing was installed, and nothing was certain.

# Chapter Two: The Payoff
Two chapters later, the machine could read aloud. [pause 400ms] The end.
"""

payload = {
    "text": SCRIPT,
    "default_voice": NARRATOR_ID,
    "format": "m4b",
    "metadata": {"title": "The Colab Miniature", "author": "VoiceStudio",
                 "narrator": "Colab Narrator"},
}

check_backend()
done = None
with requests.post(f"{BASE}/audiobook", json=payload,
                   stream=True, timeout=(10, 1800)) as r:
    if r.status_code != 200:
        api_fail(r, "POST /audiobook")
    for line in r.iter_lines(decode_unicode=True):
        if not line or not line.startswith("data:"):
            continue
        evt = _json.loads(line[len("data:"):].strip())
        if evt.get("type") == "error":
            raise SystemExit(f"Audiobook render failed: {evt}\n{LOG_HINT}")
        if evt.get("type") == "done":
            done = evt
        else:
            print(f"  [{evt.get('type')}] {evt.get('title') or ''}")

if not done:
    raise SystemExit(f"Stream ended without a 'done' event. {LOG_HINT}")
book_path = os.path.join("/content/omnivoice_data/outputs", done["output"])
print(f"Done: {done.get('chapters')} chapters, {done.get('duration_s')}s -> {book_path}")

# Inline playback: IPython's Audio widget doesn't know .m4b, so embed the
# (small) file as an HTML5 audio tag — Chrome plays AAC-in-MP4 natively.
b64 = base64.b64encode(open(book_path, "rb").read()).decode("ascii")
display(HTML(f'<audio controls src="data:audio/mp4;base64,{b64}"></audio>'))
print("Chapter markers show up in audiobook players (Apple Books, etc.) — "
      "download the file from the Colab file browser to try it.")


### 18. Video dubbing — heavy, optional

The flagship pipeline, kept honest and miniature: a 6-second synthetic clip (color frame + the cell-9 English narration) is dubbed into Spanish — upload → prep (audio extract + Demucs vocal separation) → transcribe → translate → voice-cloned TTS → mux — all through the same job API the app uses.

Only run this if you have 5 to 15 minutes to spare. Run cell 12a first (cell 13 itself is optional). The first dub also downloads the Demucs separation model. Translation goes through the free Google web endpoint via `deep-translator`, which the cell installs itself. For a fully offline dub, pass `provider="nllb"` instead (one-time download of about 2.5 GB).

In [ ]:
# ── 18. Video dubbing (mini): English clip -> Spanish dub ───────────────────
# Needs the model from cell 12a; stop early with a clear pointer if it is missing.
if not globals().get("ASR_MODEL_READY"):
    raise SystemExit("Run cell 12a (install the transcription model) first.")
import subprocess
import sys
import time

try:
    ensure_wav
except NameError:
    raise SystemExit("Run cell 8 (feature-tour helpers) first.")

from IPython.display import Video

check_backend()
TARGET_LANG, TARGET_LABEL = "es", "Spanish"

# deep-translator backs the default "google" translation provider (free web
# endpoint, no key). Offline alternative: provider="nllb" below (~2.5 GB model).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "deep-translator"], check=True)

# 18a. Build a tiny source video: a colored frame carrying the EN narration.
src_wav = ensure_wav("tts_en.wav", EN_TEXT)
src_mp4 = os.path.join(DEMO_DIR, "dub_source.mp4")
if not os.path.exists(src_mp4):
    subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
                    "-f", "lavfi", "-i", "color=c=steelblue:s=640x360:r=25",
                    "-i", src_wav, "-shortest",
                    "-c:v", "libx264", "-pix_fmt", "yuv420p", "-c:a", "aac",
                    src_mp4], check=True)
print(f"Source clip: {src_mp4} ({os.path.getsize(src_mp4)} bytes)")

def poll_task(task_id, what, timeout_s=1200):
    """Poll the persisted job row until the background task finishes."""
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        r = requests.get(f"{BASE}/jobs/{task_id}", timeout=30)
        status = r.json().get("status") if r.status_code == 200 else "pending"
        if status == "done":
            print(f" {what}: done")
            return
        if status in ("failed", "cancelled"):
            raise SystemExit(f"{what} {status}: {r.text[:1500]}\n{LOG_HINT}")
        print(".", end="", flush=True)
        time.sleep(5)
    raise SystemExit(f"{what} timed out after {timeout_s}s. {LOG_HINT}")

# 18b. Upload -> prep task (audio extract + Demucs separation; model download
#      on first run).
with open(src_mp4, "rb") as f:
    r = requests.post(f"{BASE}/dub/upload",
                      files={"video": ("dub_source.mp4", f, "video/mp4")},
                      data={"input_type": "video"}, timeout=300)
if r.status_code != 202:
    api_fail(r, "POST /dub/upload")
job_id, prep_task = r.json()["job_id"], r.json()["task_id"]
print(f"Dub job {job_id} — preparing (Demucs separation; first run downloads its model)")
poll_task(prep_task, "prep")

# 18c. Transcribe the source audio (synchronous endpoint).
print("Transcribing source audio...")
r = requests.post(f"{BASE}/dub/transcribe/{job_id}", timeout=1800)
if r.status_code != 200:
    api_fail(r, "POST /dub/transcribe")
segments = r.json()["segments"]
for s in segments:
    print(f"  [{s['start']:.2f}-{s['end']:.2f}s] {s['text']}")

# 18d. Translate the segments.
r = requests.post(f"{BASE}/dub/translate", json={
    "job_id": job_id,
    "target_lang": TARGET_LANG,
    "provider": "google",   # or "nllb" for the fully offline translator
    "segments": [{"id": str(s["id"]), "text": s["text"],
                  "start": s["start"], "end": s["end"]} for s in segments],
}, timeout=600)
if r.status_code != 200:
    api_fail(r, "POST /dub/translate")
translated = {t["id"]: t["text"] for t in r.json()["translated"]}
for sid, text in translated.items():
    print(f"  {TARGET_LANG} #{sid}: {text}")

# 18e. Generate the dubbed track — TTS voice-cloned from the source speaker.
r = requests.post(f"{BASE}/dub/generate/{job_id}", json={
    "language": TARGET_LABEL,
    "language_code": TARGET_LANG,
    "segments": [{"start": s["start"], "end": s["end"],
                  "text": translated.get(str(s["id"]), s["text"])}
                 for s in segments],
}, timeout=120)
if r.status_code != 200:
    api_fail(r, "POST /dub/generate")
print(f"Rendering dubbed track (task {r.json()['task_id']})")
poll_task(r.json()["task_id"], "dub generate")

# 18f. Mux and fetch the dubbed video.
r = requests.get(f"{BASE}/dub/download/{job_id}",
                 params={"include_tracks": TARGET_LANG, "default_track": TARGET_LANG},
                 timeout=1800)
if r.status_code != 200:
    api_fail(r, "GET /dub/download")
dub_mp4 = os.path.join(DEMO_DIR, f"dubbed_{TARGET_LANG}.mp4")
with open(dub_mp4, "wb") as f:
    f.write(r.content)
print(f"Dubbed video: {dub_mp4} ({os.path.getsize(dub_mp4)} bytes)")
display(Video(dub_mp4, embed=True, width=480))
DUB_JOB_ID = job_id  # cell 19 (vocal isolation) reuses this job


### 19. Vocal isolation (stems)

Dub prep already ran Demucs source separation, so the stems exist as job artifacts: `GET /dub/export-stems/{job_id}` returns the dubbed vocals and the original background as a zip, played here separately. Requires cell 18's job. (There is no job-free standalone isolation endpoint — separation rides the dub pipeline, so this cell does too.) Expected runtime: seconds.


In [ ]:
# ── 19. Vocal isolation: fetch and play the separated stems ─────────────────
import io
import zipfile

try:
    DUB_JOB_ID
except NameError:
    raise SystemExit("Run cell 18 (video dubbing) first — the stems are artifacts of its job.")

r = requests.get(f"{BASE}/dub/export-stems/{DUB_JOB_ID}", timeout=300)
if r.status_code != 200:
    api_fail(r, "GET /dub/export-stems")

stems_dir = os.path.join(DEMO_DIR, "stems")
os.makedirs(stems_dir, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
    names = zf.namelist()
    zf.extractall(stems_dir)
print("Stems:", names)
for name in names:
    play(os.path.join(stems_dir, name), name)


### 20. Everything you just generated

A closing inventory: every artifact the tour produced, with paths and sizes. Remember Colab storage is ephemeral — download anything you want to keep (file browser in the left sidebar), or mount Drive as shown in the troubleshooting section.


In [ ]:
# ── 20. Tour summary: artifacts on disk ─────────────────────────────────────
import os

def _ls(root, label):
    print(f"\n{label} ({root})")
    if not os.path.isdir(root):
        print("  (nothing here — the cells that write to it were not run)")
        return
    for dirpath, _dirs, files in sorted(os.walk(root)):
        for name in sorted(files):
            p = os.path.join(dirpath, name)
            print(f"  {os.path.getsize(p):>10,} B  {os.path.relpath(p, root)}")

_ls("/content/omnivoice_demos", "Feature-tour artifacts")
_ls("/content/omnivoice_data/outputs", "Backend outputs (history, stories, audiobooks)")

try:
    r = requests.get(f"{BASE}/profiles", timeout=15)
    if r.status_code == 200:
        print(f"\nSaved voice profiles: "
              f"{', '.join(p['name'] for p in r.json()) or '(none)'}")
except Exception:
    pass  # backend may already be stopped — the file listing above still stands


## Part 3 — Troubleshooting

**"device": "cpu" in /health, or generation is crawling** — the runtime has no GPU. `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. Colab's free tier sometimes has no GPUs available; trying again later usually works.

**First generation takes minutes** — that's the default model (a few GB) downloading and loading. Run cell 4 to front-load the download; after the first generation the model stays warm and requests take seconds.

**Session limits (free tier)** — Colab disconnects idle sessions and caps total runtime (up to ~12 h, often less). When the VM is recycled, **everything under `/content` is deleted**: the repo, models, and your `omnivoice_data` (voices, projects, generated audio). To keep your data across sessions, mount Google Drive and point the data dir at it *before* running cell 5:

```python
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['OMNIVOICE_DATA_DIR'] = '/content/drive/MyDrive/omnivoice_data'
```

**VRAM** — a T4 has 16 GB: plenty for TTS, cloning, and voice design. Full dubbing pipelines (separation + transcription + diarization + TTS) run closer to the limit; the app evicts idle engines automatically, but very long videos may need shorter chunks.

**The UI loads but every action fails with 403** — the backend only accepts browser calls from origins it knows. Cell 5 trusts this session's Colab proxy origin when it starts the backend, so a backend started some other way (or before this fix) refuses the proxied UI. Restart it: `!kill $(lsof -t -i:3900)`, then re-run cell 5 and cell 6. Cell 5 must run in the browser (it asks Colab for the proxy URL).

**UI tab is blank or errors out** — re-run cell 5 (it verifies `/health` without restarting a healthy backend), then cell 6. Pop-ups must be allowed for `colab.research.google.com`.

**Feature-tour cells fail with `NameError` / "Run cell 8 first"** — the tour's helper functions live in cell 8; run it once after every kernel restart.

**Backend logs** — `/content/omnivoice_backend.log`. The launch, smoke-test, and feature cells print the relevant tail or point here on failure; attach the full file when reporting an issue.

**Reporting issues** — this notebook is maintained in the main repo: [debpalash/VoiceStudio/issues](https://github.com/debpalash/VoiceStudio/issues).
